In [1]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

# Checking the file PATH
print(os.getcwd())
print(os.path.exists("./tutorial_data/tomo_all_frames.rec"))
print(os.path.exists("./tutorial_project/refined_tomograms/tomo_even_frames+tomo_odd_frames_refined.rec"))

C:\Users\chris\Desktop\Fabio\semester-project-deepdewedge\notebooks\tutorial
True
False


In [2]:
from ddw.utils.mrctools import load_mrc_data
#ddw function which loads us correct .rec and .mrc files


# Input / baseline tomogram from filtered back-projection
tomo_input = load_mrc_data("./tutorial_data/tomo_all_frames.rec")

# DeepDeWedge refined tomogram from Step 3
tomo_refined = load_mrc_data(
    "./tutorial_project/refined_tomograms/tomo_even_frames+tomo_odd_frames_refined.rec"
)

print(type(tomo_input), tomo_input.shape)
print(type(tomo_refined), tomo_refined.shape)

C:\Users\chris\anaconda3\envs\ddw_env\lib\site-packages\lightning_utilities\core\imports.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


FileNotFoundError: [Errno 2] No such file or directory: './tutorial_project/refined_tomograms/tomo_even_frames+tomo_odd_frames_refined.rec'

In [ ]:
# changing torch vectors into numpy vectors for handling np.fft later on in the notebook
if isinstance(tomo_input, torch.Tensor):
    tomo_input = tomo_input.cpu().numpy()

# changing torch vectors into numpy vectors for handling np.fft later on in the notebook
if isinstance(tomo_refined, torch.Tensor):
    tomo_refined = tomo_refined.cpu().numpy()

print(type(tomo_input), tomo_input.shape)
print(type(tomo_refined), tomo_refined.shape)

In [ ]:
z_mid = tomo_input.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

vmin = -3 * tomo_input.std()
vmax =  3 * tomo_input.std()

axes[0].imshow(tomo_input[z_mid, :, :], cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("Input FBP - XY slice")
axes[0].axis("off")

axes[1].imshow(tomo_refined[z_mid, :, :], cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title("DeepDeWedge refined - XY slice")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### Power Spectrum Analysis

In [ ]:
def compute_power_spectrum(volume):
    """
    Compute centered 3D power spectrum of a real-space volume.

    Input:
        volume: 3D numpy array, shape usually (z, y, x)

    Output:
        power: 3D numpy array, same shape, centered Fourier power spectrum
    """
    # 3D Fourier transform
    fft_volume = np.fft.fftn(volume)

    # Shift zero frequency to the center for visualization
    fft_volume_shifted = np.fft.fftshift(fft_volume)

    # Power spectrum = squared magnitude of Fourier coefficients
    power = np.abs(fft_volume_shifted) ** 2

    return power


power_input = compute_power_spectrum(tomo_input)
power_refined = compute_power_spectrum(tomo_refined)

print(power_input.shape)
print(power_refined.shape)

In [ ]:
# Central Fourier XZ slice: fix y at the center
y_mid = power_input.shape[1] // 2

power_input_xz = power_input[:, y_mid, :]
power_refined_xz = power_refined[:, y_mid, :]

# Log scale because Fourier power has a huge dynamic range
log_power_input_xz = np.log1p(power_input_xz) #np.log1p(power) is similar to log(1 + power) makes the weak fourier regions visible
log_power_refined_xz = np.log1p(power_refined_xz)


# Use same contrast range for fair comparison
vmin = min(log_power_input_xz.min(), log_power_refined_xz.min())
vmax = max(log_power_input_xz.max(), log_power_refined_xz.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(log_power_input_xz, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("Input FBP\ncentral XZ Fourier power slice")
axes[0].axis("off")

axes[1].imshow(log_power_refined_xz, cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title("DeepDeWedge refined\ncentral XZ Fourier power slice")
axes[1].axis("off")


os.makedirs("../../figures", exist_ok=True)
save_path = "../../figures/day5_power_spectrum_before_after.png"


plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Fourier Shell Correlation
def compute_fsc(vol1, vol2, num_shells=None):
    """
    Compute Fourier Shell Correlation between two 3D volumes.

    Parameters
    ----------
    vol1, vol2 : np.ndarray
        3D volumes with the same shape, usually ordered as (z, y, x).
    num_shells : int or None
        Number of radial Fourier shells. If None, use half the smallest dimension.

    Returns
    -------
    shell_frequencies : np.ndarray
        Normalized shell frequencies from low to high frequency.
    fsc_values : np.ndarray
        FSC values for each shell.
    """
    assert vol1.shape == vol2.shape, "Volumes must have the same shape."

    # Fourier transform and shift zero frequency to center
    F1 = np.fft.fftshift(np.fft.fftn(vol1))
    F2 = np.fft.fftshift(np.fft.fftn(vol2))

    shape = vol1.shape

    if num_shells is None:
        num_shells = min(shape) // 2

    # Create coordinate grid in Fourier space
    z, y, x = np.indices(shape)
    center = np.array(shape) // 2

    # Radial distance from Fourier center
    r = np.sqrt(
        (z - center[0])**2 +
        (y - center[1])**2 +
        (x - center[2])**2
    )

    r_max = min(shape) // 2
    shell_edges = np.linspace(0, r_max, num_shells + 1)

    shell_frequencies = []
    fsc_values = []

    for i in range(num_shells):
        shell_mask = (r >= shell_edges[i]) & (r < shell_edges[i + 1])

        if np.sum(shell_mask) == 0:
            shell_frequencies.append(np.nan)
            fsc_values.append(np.nan)
            continue

        F1_shell = F1[shell_mask]
        F2_shell = F2[shell_mask]

        numerator = np.sum(F1_shell * np.conj(F2_shell))
        denominator = np.sqrt(
            np.sum(np.abs(F1_shell)**2) *
            np.sum(np.abs(F2_shell)**2)
        )

        if denominator == 0:
            fsc = np.nan
        else:
            fsc = np.real(numerator / denominator)

        shell_center = 0.5 * (shell_edges[i] + shell_edges[i + 1])
        normalized_frequency = shell_center / r_max

        shell_frequencies.append(normalized_frequency)
        fsc_values.append(fsc)

    return np.array(shell_frequencies), np.array(fsc_values)

In [ ]:
tomo_even = load_mrc_data("./tutorial_data/tomo_even_frames.rec")
tomo_odd = load_mrc_data("./tutorial_data/tomo_odd_frames.rec")

if isinstance(tomo_even, torch.Tensor):
    tomo_even = tomo_even.cpu().numpy()

if isinstance(tomo_odd, torch.Tensor):
    tomo_odd = tomo_odd.cpu().numpy()

print(tomo_even.shape, tomo_odd.shape)

### Voxel size and FSC resolution estimate

The FSC curve gives a **threshold frequency**. To report this as a real-space resolution, we need the voxel size from the MRC/REC header.

If the normalized frequency is defined such that `1.0 = Nyquist frequency`, then:

```text
resolution [voxels] = 2 / f_norm
resolution [Å]      = 2 * voxel_size [Å] / f_norm
```


In [ ]:
from pathlib import Path
import numpy as np

def read_voxel_size_angstrom(mrc_path):
    """
    Read voxel size from an MRC/REC header.

    Returns
    -------
    voxel_size_angstrom : float or None
        Returns a single voxel size if x, y, z are equal.
        Returns None if the header does not contain a usable voxel size.
    """
    try:
        import mrcfile
    except ImportError:
        raise ImportError(
            "mrcfile is needed to read voxel size. Install with: pip install mrcfile"
        )

    mrc_path = Path(mrc_path)
    with mrcfile.open(mrc_path, permissive=True) as mrc:
        vx = float(mrc.voxel_size.x)
        vy = float(mrc.voxel_size.y)
        vz = float(mrc.voxel_size.z)

    print(f"{mrc_path}")
    print(f"voxel size from header: x={vx:.6g} Å, y={vy:.6g} Å, z={vz:.6g} Å")

    if vx <= 0 or vy <= 0 or vz <= 0:
        print("Voxel size in header is missing or invalid.")
        return None

    if not np.allclose([vx, vy, vz], vx):
        print("Warning: voxel size is anisotropic. For FSC, using the mean voxel size.")
        return float(np.mean([vx, vy, vz]))

    return vx


voxel_size_A = read_voxel_size_angstrom("./tutorial_data/tomo_even_frames.rec")
print("voxel_size_A =", voxel_size_A)


In [ ]:
freqs, fsc_values = compute_fsc(tomo_even, tomo_odd)

plt.figure(figsize=(7, 5))
plt.plot(freqs, fsc_values, label="FBP even vs odd")
plt.axhline(0.143, linestyle="--", label="0.143 criterion")
plt.axhline(0.5, linestyle="--", label="0.5 criterion")

plt.xlabel("Normalized spatial frequency")
plt.ylabel("FSC")
plt.title("Fourier Shell Correlation: even vs odd FBP")
plt.ylim(-0.1, 1.05)
plt.legend()
plt.tight_layout()

os.makedirs("../../figures", exist_ok=True)
save_path = "../../figures/day5_fsc_even_odd_fbp.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved to:", os.path.abspath(save_path))
print("File exists:", os.path.exists(save_path))

### Convert FSC threshold crossing to resolution

Here we estimate where the FSC curve first crosses a threshold, then convert the corresponding normalized spatial frequency into a resolution.

Interpretation: the reported resolution is the smallest real-space length scale that is still reproducibly supported by the two independent reconstructions according to the chosen FSC criterion.


In [ ]:
def estimate_fsc_resolution(freqs, fsc_values, threshold=0.5, voxel_size_angstrom=None):
    """
    Estimate resolution from the FSC threshold crossing.

    Parameters
    ----------
    freqs : array
        Normalized spatial frequencies, where 1.0 corresponds to Nyquist.
    fsc_values : array
        FSC values.
    threshold : float
        FSC threshold, e.g. 0.5 or 0.143.
    voxel_size_angstrom : float or None
        Voxel size in Angstrom. If None, only resolution in voxels is returned.

    Returns
    -------
    result : dict
        Contains threshold, crossing_frequency, resolution_voxels,
        and resolution_angstrom.
    """
    freqs = np.asarray(freqs, dtype=float)
    fsc_values = np.asarray(fsc_values, dtype=float)

    valid = np.isfinite(freqs) & np.isfinite(fsc_values)
    freqs = freqs[valid]
    fsc_values = fsc_values[valid]

    # Find the first point where FSC drops below the threshold
    below = np.where(fsc_values < threshold)[0]

    if len(below) == 0:
        return {
            "threshold": threshold,
            "crossing_frequency": np.nan,
            "resolution_voxels": np.nan,
            "resolution_angstrom": np.nan,
            "note": f"FSC never drops below {threshold} within sampled frequencies."
        }

    idx = below[0]

    if idx == 0:
        f_cross = freqs[0]
    else:
        # Linear interpolation between the last point above and first point below
        f1, f2 = freqs[idx - 1], freqs[idx]
        y1, y2 = fsc_values[idx - 1], fsc_values[idx]

        if y2 == y1:
            f_cross = f2
        else:
            f_cross = f1 + (threshold - y1) * (f2 - f1) / (y2 - y1)

    resolution_voxels = 2.0 / f_cross if f_cross > 0 else np.nan

    if voxel_size_angstrom is None:
        resolution_angstrom = np.nan
    else:
        resolution_angstrom = resolution_voxels * voxel_size_angstrom

    return {
        "threshold": threshold,
        "crossing_frequency": f_cross,
        "resolution_voxels": resolution_voxels,
        "resolution_angstrom": resolution_angstrom,
        "note": "ok"
    }


res_05 = estimate_fsc_resolution(freqs, fsc_values, threshold=0.5, voxel_size_angstrom=voxel_size_A)
res_0143 = estimate_fsc_resolution(freqs, fsc_values, threshold=0.143, voxel_size_angstrom=voxel_size_A)

for res in [res_05, res_0143]:
    print("\nFSC threshold:", res["threshold"])
    print("crossing frequency:", res["crossing_frequency"])
    print("resolution [voxels]:", res["resolution_voxels"])
    print("resolution [Å]:", res["resolution_angstrom"])
    print("note:", res["note"])


In [ ]:
# Plot FSC again and mark the FSC=0.5 resolution estimate
plt.figure(figsize=(7, 5))
plt.plot(freqs, fsc_values, label="FBP even vs odd")

plt.axhline(0.5, linestyle="--", label="FSC = 0.5")
plt.axhline(0.143, linestyle="--", label="FSC = 0.143")

if np.isfinite(res_05["crossing_frequency"]):
    plt.axvline(res_05["crossing_frequency"], linestyle=":", label="0.5 crossing")
    title_res = f"FSC 0.5 resolution ≈ {res_05['resolution_voxels']:.2f} voxels"
    if np.isfinite(res_05["resolution_angstrom"]):
        title_res += f" ≈ {res_05['resolution_angstrom']:.1f} Å"
else:
    title_res = "FSC 0.5 crossing not found"

plt.xlabel("Normalized spatial frequency")
plt.ylabel("FSC")
plt.title(title_res)
plt.ylim(-0.1, 1.05)
plt.legend()
plt.tight_layout()

os.makedirs("../../figures", exist_ok=True)
save_path = "../../figures/day5_fsc_even_odd_resolution.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved to:", os.path.abspath(save_path))
print("File exists:", os.path.exists(save_path))
